# Validación Académica: Modelo de Inversión Mixta (TCO a 5 años)

Este notebook valida el modelo de inversión mixta de la flota (Diésel + Eléctricos). Muestra los cálculos del TCO (Total Cost of Ownership) para 1 camión y para la flota consolidada bajo 3 modalidades: Compra, Leasing Financiero y Renting Operativo.

In [25]:
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import plotly.graph_objects as go
import plotly.express as px
import pandas as pd

from logistic_core.utils.investment_analyzer import InvestmentAnalyzer
from logistic_core.config import (
    DIESEL_CAPEX, EV_CAPEX, EV_MOVES_AYUDA,
    FLEET_MIX_DIESEL, FLEET_MIX_EV,
    KMS_ANUALES_POR_CAMION, TCO_HORIZON_YEARS,
    TCO_WACC, TCO_INFLACION_ANUAL, TCO_TAX_RATE,
    DIESEL_RENTING_MENSUAL, DIESEL_LEASING_MENSUAL,
    EV_RENTING_MENSUAL, EV_LEASING_MENSUAL
)

## 1. Parámetros de Entrada

Basados en datos de mercado 2025-2026.

In [26]:
print("--- PARÁMETROS DE ENTRADA ---")
print(f"  Horizonte:          {TCO_HORIZON_YEARS} años")
print(f"  Km anuales/camión:  {KMS_ANUALES_POR_CAMION:_} km")
print(f"  WACC:               {TCO_WACC*100:.1f}%")
print(f"  Inflación:          {TCO_INFLACION_ANUAL*100:.1f}%")
print(f"  Tipo IS:            {TCO_TAX_RATE*100:.0f}%")
print(f"  Composición Flota:  {FLEET_MIX_DIESEL} Diésel + {FLEET_MIX_EV} BEV = {FLEET_MIX_DIESEL + FLEET_MIX_EV} camiones")
print(f"  CAPEX Diésel:       {DIESEL_CAPEX:_} €")
print(f"  CAPEX BEV:          {EV_CAPEX:_} € (MOVES: -{EV_MOVES_AYUDA:_} €)")

--- PARÁMETROS DE ENTRADA ---
  Horizonte:          5 años
  Km anuales/camión:  130_000 km
  WACC:               7.0%
  Inflación:          2.0%
  Tipo IS:            25%
  Composición Flota:  7 Diésel + 4 BEV = 11 camiones
  CAPEX Diésel:       140_000 €
  CAPEX BEV:          340_000 € (MOVES: -90_000 €)


## 2. Modelización Matemática (TCO Unitario)

El cálculo del TCO se desglosa por modalidad de adquisición.

### A. Modalidad de Compra

$$TCO_{compra} = -CAPEX + \sum_{t=1}^{5} \frac{-OPEX_t + (OPEX_t + Amort_t) \times \tau}{(1+WACC)^t} + \frac{VR \times (1-\tau)}{(1+WACC)^5}$$

### B. Modalidad de Leasing Financiero

$$TCO_{leasing} = \sum_{t=1}^{5} \frac{-OPEX_t - C_{lease,t} + (OPEX_t + C_{lease,t}) \times \tau}{(1+WACC)^t}$$

### C. Modalidad de Renting Operativo (Full-Service)

$$TCO_{renting} = \sum_{t=1}^{5} \frac{-E_t - C_{rent,t} + (E_t + C_{rent,t}) \times \tau}{(1+WACC)^t}$$

Donde:
*   $OPEX_t$ incluye Energía ($E_t$), Mantenimiento ($M_t$) y Seguro ($S_t$).
*   $\tau$ es el impuesto de sociedades (25%).
*   $WACC$ es la tasa de descuento (7%).

In [27]:
analyzer = InvestmentAnalyzer()
tabla = analyzer.generar_tabla_comparativa()

print("--- TCO UNITARIO (1 CAMIÓN) ---")
print(f"{'':35} | {'COMPRA':>15} | {'LEASING':>15} | {'RENTING':>15}")
print("-" * 87)

for tec, label in [("diesel", "Diésel (Euro VI)"), ("electrico", "Eléctrico (BEV 44t)")]:
    vals = []
    for mod in ["compra", "leasing", "renting"]:
        v = tabla["por_camion"][tec][mod]["tco_van_acumulado"]
        vals.append(f"{abs(v):>12_.0f} €")
    print(f"  {label:<33} | {vals[0]:>15} | {vals[1]:>15} | {vals[2]:>15}")

print()

for tec, label in [("diesel", "Diésel (€/km)"), ("electrico", "BEV (€/km)")]:
    vals = []
    for mod in ["compra", "leasing", "renting"]:
        v = tabla["por_camion"][tec][mod]["coste_neto_por_km"]
        vals.append(f"{v:>12.3f}")
    print(f"  {label:<33} | {vals[0]:>15} | {vals[1]:>15} | {vals[2]:>15}")

--- TCO UNITARIO (1 CAMIÓN) ---
                                    |          COMPRA |         LEASING |         RENTING
---------------------------------------------------------------------------------------
  Diésel (Euro VI)                  |       356_737 € |       363_842 € |       309_034 €
  Eléctrico (BEV 44t)               |       288_290 € |       255_336 € |       305_695 €

  Diésel (€/km)                     |           0.549 |           0.560 |           0.475
  BEV (€/km)                        |           0.444 |           0.393 |           0.470


In [28]:
# Visualización Plotly: TCO Unitario
data_plot = []
for tec, label in [("diesel", "Diésel (Euro VI)"), ("electrico", "Eléctrico (BEV 44t)")]:
    for mod in ["compra", "leasing", "renting"]:
        v = abs(tabla["por_camion"][tec][mod]["tco_van_acumulado"])
        data_plot.append({"Tecnología": label, "Modalidad": mod.capitalize(), "TCO (€)": v})

df_unit = pd.DataFrame(data_plot)
fig1 = px.bar(df_unit, x="Tecnología", y="TCO (€)", color="Modalidad", barmode="group",
             title="TCO Unitario por Tecnología y Modalidad (5 años)",
             color_discrete_sequence=px.colors.qualitative.Pastel)
fig1.update_layout(yaxis_tickformat=".0f")
fig1.show()

## 3. Flujos de Caja Detallados (Modalidad: Compra)

In [29]:
print("--- FLUJOS DE CAJA DETALLADOS (Compra) ---")
print(f"{'Año':<6} | {'Diésel':>15} | {'Eléctrico':>15}")
print("-" * 42)
fd = tabla["por_camion"]["diesel"]["compra"]["flujos_anuales"]
fe = tabla["por_camion"]["electrico"]["compra"]["flujos_anuales"]
for t in range(len(fd)):
    d_val = f"{fd[t]:>12_.0f} €"
    e_val = f"{fe[t]:>12_.0f} €"
    print(f"  t={t:<3} | {d_val:>15} | {e_val:>15}")

--- FLUJOS DE CAJA DETALLADOS (Compra) ---
Año    |          Diésel |       Eléctrico
------------------------------------------
  t=0   |      -140_000 € |      -340_000 €
  t=1   |       -55_944 € |        72_330 €
  t=2   |       -57_203 € |       -18_363 €
  t=3   |       -58_487 € |       -19_070 €
  t=4   |       -59_797 € |       -19_792 €
  t=5   |       -29_633 € |        43_222 €


## 4. TCO Consolidado: Flota Mixta

Se escala el modelo unitario a la composición real de la flota (7 Diesel + 4 EV).

$$TCO_{flota}^{mod} = N_{diesel} \times TCO_{diesel}^{mod} + N_{ev} \times TCO_{ev}^{mod}$$

In [30]:
print("--- TCO CONSOLIDADO FLOTA MIXTA ---")
print(f"{'MODALIDAD':<18} | {'DIÉSEL (7x)':>15} | {'BEV (4x)':>15} | {'TOTAL':>15} | {'€/km':>8}")
print("-" * 80)

for mod, label in [("compra", "Compra"), ("leasing", "Leasing"), ("renting", "Renting")]:
    m = tabla["flota_mixta"][mod]
    is_best = (mod == tabla["recomendacion"]["modalidad"])
    marker = " <-- ÁPTIMO" if is_best else ""
    d = f"{abs(m['tco_diesel_subtotal']):>12_.0f}"
    e = f"{abs(m['tco_ev_subtotal']):>12_.0f}"
    t = f"{abs(m['tco_total']):>12_.0f}"
    k = f"{m['coste_km_medio']:.3f}"
    print(f"  {label:<16} | {d:>15} | {e:>15} | {t:>15} | {k:>8}{marker}")

rec = tabla["recomendacion"]
print("-" * 80)
print(f"\n  RECOMENDACIÓN: {rec['modalidad'].upper()}")
print(f"  Ahorro vs {rec['vs_modalidad'].upper()}: {rec['ahorro_vs_peor_eur']:_} € ({rec['ahorro_pct']:.1f}%)")

--- TCO CONSOLIDADO FLOTA MIXTA ---
MODALIDAD          |     DIÉSEL (7x) |        BEV (4x) |           TOTAL |     €/km
--------------------------------------------------------------------------------
  Compra           |       2_497_160 |       1_153_161 |       3_650_321 |    0.511
  Leasing          |       2_546_896 |       1_021_343 |       3_568_238 |    0.499
  Renting          |       2_163_241 |       1_222_781 |       3_386_022 |    0.474 <-- ÁPTIMO
--------------------------------------------------------------------------------

  RECOMENDACIÓN: RENTING
  Ahorro vs COMPRA: 264_299.3135625492 € (7.2%)


In [31]:
# Visualización Plotly: Consolidado Mixto
data_mixed = []
for mod in ["compra", "leasing", "renting"]:
    data_mixed.append({"Modalidad": mod.capitalize(), "Tipo": "Diésel (7)", "TCO (€)": abs(tabla["flota_mixta"][mod]["tco_diesel_subtotal"])})
    data_mixed.append({"Modalidad": mod.capitalize(), "Tipo": "BEV (4)", "TCO (€)": abs(tabla["flota_mixta"][mod]["tco_ev_subtotal"])})

df_mixed = pd.DataFrame(data_mixed)

fig2 = px.bar(df_mixed, x="Modalidad", y="TCO (€)", color="Tipo", barmode="stack",
             title="TCO Consolidado Flota Mixta (11 Camiones) por Modalidad",
             color_discrete_sequence=px.colors.qualitative.Dark24)
fig2.update_layout(yaxis_tickformat=".0f")
fig2.show()

## 5. Análisis de Sensibilidad: Sin Subvención MOVES

In [32]:
print("--- SENSIBILIDAD: SIN SUBVENCIÓN MOVES ---")
analyzer_no_moves = InvestmentAnalyzer()
# Forzamos la ayuda a 0 para ver el impacto empírico en el BEV.
analyzer_no_moves.params["electrico"]["ayuda_moves"] = 0
tabla_no_moves = analyzer_no_moves.generar_tabla_comparativa()

for mod, label in [("compra", "Compra"), ("leasing", "Leasing"), ("renting", "Renting")]:
    m_con = tabla["flota_mixta"][mod]
    m_sin = tabla_no_moves["flota_mixta"][mod]
    delta = abs(m_sin["tco_total"]) - abs(m_con["tco_total"])
    print(f"  {label:<16}: TCO sin MOVES = {abs(m_sin['tco_total']):>12_.0f} €"
          f" (incremento: +{delta:_} €)")

--- SENSIBILIDAD: SIN SUBVENCIÓN MOVES ---
  Compra          : TCO sin MOVES =    3_986_770 € (incremento: +336_448.59813084174 €)
  Leasing         : TCO sin MOVES =    3_904_687 € (incremento: +336_448.5981308408 €)
  Renting         : TCO sin MOVES =    3_386_022 € (incremento: +0.0 €)
